In [1]:
# === Factor Analysis pipeline with optimal-factor selection (PA + MAP) ===
# Requires: pandas, numpy, scikit-learn

import pandas as pd
import numpy as np
import re
from sklearn.decomposition import FactorAnalysis
from sklearn.preprocessing import StandardScaler
from pathlib import Path

# ---------- 0) Load ----------
csv_path = Path("/data/demo_data/16_personality/16P.csv")

# Handle non-UTF8 encodings gracefully
for enc in ["utf-8", "utf-8-sig", "cp1252", "latin1"]:
    try:
        df_raw = pd.read_csv(csv_path, encoding=enc)
        break
    except Exception:
        continue

# Keep numeric items only (drop string labels like ‘Personality’ and any ID)
label_col = "Personality" if "Personality" in df_raw.columns else None
data = df_raw.copy()
for c in data.columns:
    if c != label_col:
        data[c] = pd.to_numeric(data[c], errors="coerce")

X = data.drop(columns=[label_col] if label_col in data.columns else [])
if "Response Id" in X.columns:
    X = X.drop(columns=["Response Id"])

# Remove near-constant columns and impute missing with column means
stds = X.std(numeric_only=True)
const_cols = stds[stds < 1e-6].index.tolist()
if const_cols:
    X = X.drop(columns=const_cols)
X = X.apply(lambda s: s.fillna(s.mean()), axis=0)

# Standardize (work on correlation matrix scale)
Z = StandardScaler(with_mean=True, with_std=True).fit_transform(X.values)

# ---------- 1) Parallel Analysis (PA) ----------
def parallel_analysis(zmat, n_iter=200, random_state=123, use_pctl=95):
    rng = np.random.default_rng(random_state)
    n, p = zmat.shape
    R = np.corrcoef(zmat, rowvar=False)
    evals, _ = np.linalg.eigh(R)
    evals = np.sort(evals)[::-1]
    rand_eigs = np.zeros((n_iter, p))
    for i in range(n_iter):
        Xr = rng.standard_normal(size=(n, p))
        # z-score columns
        Xr = (Xr - Xr.mean(axis=0)) / Xr.std(axis=0, ddof=1)
        Rr = np.corrcoef(Xr, rowvar=False)
        vals, _ = np.linalg.eigh(Rr)
        rand_eigs[i, :] = np.sort(vals)[::-1]
    thresh = np.percentile(rand_eigs, use_pctl, axis=0)
    m_pa = int(np.sum(evals > thresh))
    return evals, thresh, m_pa

real_eigs, rand_thresh, m_pa = parallel_analysis(Z, n_iter=100, random_state=123, use_pctl=95)

# ---------- 2) Velicer’s MAP (principal-components residuals) ----------
def velicer_map(zmat, max_factors=30):
    R = np.corrcoef(zmat, rowvar=False)
    vals, vecs = np.linalg.eigh(R)
    idx = np.argsort(vals)[::-1]
    vals, vecs = vals[idx], vecs[:, idx]
    A = vecs @ np.diag(np.sqrt(np.maximum(vals, 0)))  # loadings for PCs
    p = R.shape[0]
    mf = min(max_factors, p-1)
    map_vals = []
    for m in range(mf+1):
        if m == 0:
            resid = R - np.eye(p)
        else:
            Rm = A[:, :m] @ A[:, :m].T
            resid = R - Rm
            np.fill_diagonal(resid, 0)
        off = resid[~np.eye(p, dtype=bool)]
        map_vals.append(np.mean(off**2))
    return int(np.argmin(map_vals)), map_vals

m_map, map_curve = velicer_map(Z, max_factors=30)

# Choose the final dimensionality (parsimonious): use MAP; keep PA alternative
m_final = m_map if m_map >= 1 else (m_pa if m_pa >= 1 else 1)

# ---------- 3) Factor Analysis + Varimax rotation ----------
def varimax(Phi, gamma=1.0, q=50, tol=1e-6):
    """
    Orthogonal varimax rotation.
    Phi: (n_vars, n_factors) unrotated loadings
    Returns: rotated loadings, rotation matrix
    """
    p, k = Phi.shape
    R = np.eye(k)
    d = 0
    for _ in range(q):
        d_old = d
        Lambda = Phi @ R
        u, s, vh = np.linalg.svd(
            Phi.T @ (Lambda**3 - (gamma/p) * (Lambda @ np.diag(np.diag(Lambda.T @ Lambda))))
        )
        R = u @ vh
        d = np.sum(s)
        if d_old != 0 and d/d_old < 1 + tol:
            break
    return Phi @ R, R

def fit_fa_and_rotate(zmat, n_factors):
    fa = FactorAnalysis(n_components=n_factors, random_state=123)
    fa.fit(zmat)
    # sklearn: components_ shape = (n_factors, n_vars)
    loadings = fa.components_.T
    L_rot, R = varimax(loadings)
    comm = (L_rot**2).sum(axis=1)
    ss = (L_rot**2).sum(axis=0)
    prop_common = ss / comm.sum()
    return L_rot, prop_common

# Fit final (MAP) solution
L_map, prop_map = fit_fa_and_rotate(Z, m_final)

# Fit PA solution as alternative
L_pa, prop_pa = fit_fa_and_rotate(Z, m_pa)

# ---------- 4) Heuristic factor naming ----------
item_names = X.columns.tolist()
name_map = {
    "Extraversion": [
        r"group", r"social", r"introduce", r"around others", r"contacts your friends",
        r"lively", r"draw(ing)? attention", r"walking up to someone"
    ],
    "Conscientiousness / Orderliness": [
        r"to-?do", r"list", r"schedule", r"organiz", r"methodical", r"plan",
        r"deadline", r"finish one project", r"chores", r"back on track", r"complete things"
    ],
    "Agreeableness / Empathy": [
        r"empath", r"care not to make people look bad", r"helping others",
        r"pass along", r"understand views", r"considerate"
    ],
    "Neuroticism / Negative Emotionality": [
        r"worr(y|ies|ying)", r"overwhelmed", r"mood", r"insecure",
        r"mistakes.*bother", r"emotions control", r"confidence", r"stress"
    ],
    "Openness / Intellect": [
        r"art|museums", r"interpret", r"philosoph", r"creative", r"controversial",
        r"intrigued", r"books? and movies?", r"theoretical"
    ],
    "Assertiveness / Dominance": [
        r"leadership", r"argue|arguing|watching people argue", r"initiat(e|es|ing)",
        r"efficient", r"lose patience", r"rationalit"
    ],
}

def propose_names(loadings):
    Ldf = pd.DataFrame(loadings, index=item_names, columns=[f"F{i+1}" for i in range(loadings.shape[1])])
    labels = []
    for j in range(Ldf.shape[1]):
        top_idx = np.abs(Ldf.iloc[:, j]).nlargest(12).index
        haystack = " ".join([s.lower() for s in top_idx])
        best_label, best_hits = None, -1
        for label, patterns in name_map.items():
            hits = sum(1 for p in patterns if re.search(p, haystack))
            if hits > best_hits:
                best_label, best_hits = label, hits
        labels.append(best_label or f"Factor {j+1}")
    return Ldf, labels

Ldf_map, labels_map = propose_names(L_map)
Ldf_pa,  labels_pa  = propose_names(L_pa)

# ---------- 5) Exports ----------
# MAP (preferred)
summary_map = pd.DataFrame({
    "Factor": [f"F{i+1}" for i in range(L_map.shape[1])],
    "SumSqLoadings": (L_map**2).sum(axis=0),
    "PropCommonVar": prop_map,
    "ProposedName": labels_map,
})
Ldf_map.to_csv("factor_loadings_rotated_MAP13.csv")
summary_map.to_csv("factor_summary_MAP13.csv", index=False)

# PA (alternative)
summary_pa = pd.DataFrame({
    "Factor": [f"F{i+1}" for i in range(L_pa.shape[1])],
    "SumSqLoadings": (L_pa**2).sum(axis=0),
    "PropCommonVar": prop_pa,
    "ProposedName": labels_pa,
})
Ldf_pa.to_csv("factor_loadings_rotated.csv")
summary_pa.to_csv("factor_summary.csv", index=False)

# Optional: show quick text summary
print(f"Parallel Analysis → {m_pa} factors")
print(f"Velicer MAP      → {m_final} factors (used)")
print("\nPreview (MAP solution):")
print(summary_map)

Parallel Analysis → 17 factors
Velicer MAP      → 13 factors (used)

Preview (MAP solution):
   Factor  SumSqLoadings  PropCommonVar                         ProposedName
0      F1       0.997775       0.093553                         Extraversion
1      F2       0.964719       0.090454                         Extraversion
2      F3       0.820187       0.076902                 Openness / Intellect
3      F4       0.757757       0.071049  Neuroticism / Negative Emotionality
4      F5       0.977303       0.091634                         Extraversion
5      F6       0.714393       0.066983      Conscientiousness / Orderliness
6      F7       0.809786       0.075927                 Openness / Intellect
7      F8       0.817621       0.076662                         Extraversion
8      F9       0.930447       0.087241      Conscientiousness / Orderliness
9     F10       0.805083       0.075486                         Extraversion
10    F11       0.684991       0.064226      Conscientiousne

Final factor names (MAP = 13)


F1 — Quiet Self‑Containment
Centered on avoiding drawing attention, lower sociability, and calm self‑regulation—i.e., a reserved interpersonal stance. 1


F2 — Planner–Improviser Dial
Contrasts backup plans and structure with last‑minute / “do what I feel” tendencies—a planning vs. spontaneity axis. 1


F3 — Head‑First Analytic Style
Emphasizes following head over heart and a cooler, less emotive social stance (e.g., discomfort with calls), suggesting analytic detachment. 1


F4 — Interpretive Openness with Mood Flux
Mix of liking ambiguous endings and some mood variability, pointing to reflective openness coupled with emotional shifts. 1


F5 — Social Boldness & Expressivity
High walking up to others, emotional resonance (crying), and lower impression concern—an outgoing, emotive social style. 1


F6 — Order Under Pressure
Reversed loadings on lists/schedules and worry/overwhelm indicate a theme of maintaining order while managing strain. 1


F7 — Wide‑Angle Curiosity
Interested in many things paired with lower group pull and self‑confident restraint—curiosity spread across options. 1


F8 — Pro‑Social Initiative
Balance of initiating with friends and helping others, with calibrated need for high‑energy settings—constructive sociability. 1


F9 — Finish‑First Task Focus
Finish one project before another and get back on track themes—sustained task completion and persistence. 1


F10 — Event‑Shy Introversion
Strong not introducing oneself at events plus quieter preferences—situational social reticence. 1


F11 — Perspective‑Taking Spontaneity
Understanding different views with a burst‑style work pattern—open‑minded yet non‑rigid execution. 1


F12 — Soft‑Hearted Sensitivity
Blend of empathy/emotional control with lower draw to bustling places—sensitivity with measured stimulation seeking. 1


F13 — Perfectionist Strain
Old mistakes still bother, deadline struggle, and tension around standards—perfectionism under pressure

In [5]:
# === Cluster analysis on MAP-derived factors ===
# Prereqs: pandas, numpy, scikit-learn

import pandas as pd, numpy as np, re
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import FactorAnalysis
from sklearn.cluster import MiniBatchKMeans
from sklearn.metrics import silhouette_score
from pathlib import Path

# ---------- 0) Load & clean ----------
fn = Path("/data/demo_data/16_personality/16P.csv")
for enc in ["utf-8", "utf-8-sig", "cp1252", "latin1"]:
    try:
        df = pd.read_csv(fn, encoding=enc)
        break
    except Exception:
        pass

label_col = "Personality" if "Personality" in df.columns else None
X = df.drop(columns=[label_col] if label_col in df.columns else []).copy()
if "Response Id" in X.columns:
    X = X.drop(columns=["Response Id"])
for c in X.columns:
    X[c] = pd.to_numeric(X[c], errors="coerce")
X = X.apply(lambda s: s.fillna(s.mean()), axis=0)

# Work on correlation scale
Z = StandardScaler(with_mean=True, with_std=True).fit_transform(X.values)

# ---------- 1) Velicer’s MAP to choose the number of factors ----------
R = np.corrcoef(Z, rowvar=False)
vals, vecs = np.linalg.eigh(R)
idx = np.argsort(vals)[::-1]
vals, vecs = vals[idx], vecs[:, idx]
A = vecs @ np.diag(np.sqrt(np.maximum(vals, 0)))

p = R.shape[0]
max_factors = min(30, p - 1)
map_vals = []
for m in range(max_factors + 1):
    if m == 0:
        resid = R - np.eye(p)
    else:
        Rm = A[:, :m] @ A[:, :m].T
        resid = R - Rm
        np.fill_diagonal(resid, 0)
    off = resid[~np.eye(p, dtype=bool)]
    map_vals.append(np.mean(off**2))
m_final = int(np.argmin(map_vals)) or 1

# ---------- 2) Factor Analysis + Varimax rotation ----------
def varimax(Phi, gamma=1.0, q=50, tol=1e-6):
    """Orthogonal varimax rotation."""
    p, k = Phi.shape
    R = np.eye(k)
    d = 0
    for _ in range(q):
        d_old = d
        Lam = Phi @ R
        u, s, vh = np.linalg.svd(Phi.T @ (Lam**3 - (gamma/p)*(Lam @ np.diag(np.diag(Lam.T @ Lam)))))
        R = u @ vh
        d = np.sum(s)
        if d_old != 0 and d/d_old < 1 + tol:
            break
    return Phi @ R, R

fa = FactorAnalysis(n_components=m_final, random_state=123)
fa.fit(Z)
L_unrot = fa.components_.T                           # (p x m)
L_rot, Rm = varimax(L_unrot)                         # rotated loadings

# ---------- 3) Regression factor scores (using rotated loadings) ----------
Psi = np.diag(fa.noise_variance_)
Psi_inv = np.linalg.inv(Psi)
B = Psi_inv @ L_rot @ np.linalg.inv(L_rot.T @ Psi_inv @ L_rot)  # (p x m)
F_scores = Z @ B                                               # (n x m)

# Standardize factor scores for clustering
Fz = StandardScaler(with_mean=True, with_std=True).fit_transform(F_scores)

# ---------- 4) Model selection & final clustering ----------
rng = np.random.RandomState(42)
sub_idx = rng.choice(Fz.shape[0], size=min(10000, Fz.shape[0]), replace=False)
X_sub = Fz[sub_idx]

best_k, best_sil = None, -1
for k in range(2, 9):
    mbk = MiniBatchKMeans(n_clusters=k, random_state=123, batch_size=4096, n_init='auto')
    labels_sub = mbk.fit_predict(X_sub)
    sil = silhouette_score(X_sub, labels_sub)
    if sil > best_sil:
        best_sil, best_k = sil, k

km = MiniBatchKMeans(n_clusters=best_k, random_state=123, batch_size=4096, n_init='auto')
labels = km.fit_predict(Fz)
centroids = km.cluster_centers_  # shape (k x m), z-scored factor space

# ---------- 5) Name clusters by dominant factor directions ----------
# Positive/negative direction labels for each factor (m_final expected 13)
pos_label = {
    0: "Quiet Self-Containment",
    1: "Deliberate Planning",
    2: "Analytic (Head-First) Style",
    3: "Interpretive Openness",
    4: "Social Boldness",
    5: "Orderliness under Pressure",
    6: "Broad Curiosity",
    7: "Pro-Social Initiative",
    8: "Finish-First Focus",
    9: "Event-Related Introversion",
    10: "Perspective-Taking",
    11: "Empathic Sensitivity",
    12: "Perfectionist Concern",
}
neg_label = {
    0: "Expressive Sociability",
    1: "Freeform Improvisation",
    2: "Warmly Intuitive Style",
    3: "Concrete Stability",
    4: "Reserved Composure",
    5: "Unstructured Under Pressure",
    6: "Focused Selectivity",
    7: "Low-initiative Sociability",
    8: "Multi-Task Drift",
    9: "Event-Ready Extraversion",
    10: "Single-Perspective Focus",
    11: "Tough-Minded Reserve",
    12: "Flexible Acceptance",
}

names, descriptions = [], []
for c in range(best_k):
    v = centroids[c]
    top = np.argsort(-np.abs(v))[:3]
    labels_dir = [pos_label[j] if v[j] >= 0 else neg_label[j] for j in top]
    name = f"{labels_dir[0]} + {labels_dir[1]}"
    names.append(name)

    strengths_list, weaknesses_list = [], []
    for j in top:
        if v[j] >= 0:
            if j in [1,5,8]: strengths_list.append("reliable execution and follow-through")
            if j in [4,7]: strengths_list.append("confident, prosocial engagement")
            if j in [6,10,3]: strengths_list.append("open-minded curiosity and perspective-taking")
            if j in [11]: strengths_list.append("interpersonal empathy and sensitivity")
            if j in [2]: strengths_list.append("clear analytical thinking")
        else:
            if j in [1,5,8]: weaknesses_list.append("inconsistency and uneven follow-through")
            if j in [4,7,9]: weaknesses_list.append("lower social initiative in group settings")
            if j in [6,10,3]: weaknesses_list.append("narrower focus and less appetite for ambiguity")
            if j in [11]: weaknesses_list.append("reduced emotional expressiveness")
            if j in [2]: weaknesses_list.append("less reliance on structured analysis")
    if np.abs(v[12]) > 0.6 and v[12] > 0:
        weaknesses_list.append("susceptibility to perfectionistic pressure and deadline strain")

    overview = f"This cluster combines {labels_dir[0].lower()}, {labels_dir[1].lower()}, and {labels_dir[2].lower()} as its most distinctive tendencies."
    strengths = "Their strengths include " + ", ".join(sorted(set(strengths_list))) + "." if strengths_list \
                else "They benefit from flexibility and the ability to adjust to varied contexts."
    weaknesses = "Potential weaknesses include " + ", ".join(sorted(set(weaknesses_list))) + "." if weaknesses_list \
                 else "Potential blind spots include over-reliance on preferred styles when contexts demand the opposite."
    descriptions.append((overview, strengths, weaknesses))

# ---------- 6) Save artifacts ----------
pd.DataFrame({"cluster": labels}).to_csv("cluster_assignments_MAP13.csv", index=False)
centroid_df = pd.DataFrame(centroids, columns=[f"F{i+1}" for i in range(centroids.shape[1])])
centroid_df.insert(0, "Cluster", range(best_k))
centroid_df["Name"] = names
centroid_df.to_csv("cluster_centroids_MAP13.csv", index=False)

print(f"Selected k = {best_k}")
print(pd.DataFrame({'Cluster': range(best_k), 'Name': names}))
print("\\nFirst two clusters (3-sentence descriptions):")
for i in range(min(2, best_k)):
    print(f"Cluster {i} — {names[i]}") 
    for line in descriptions[i]:
        print(line)

Selected k = 7
   Cluster                                               Name
0        0         Focused Selectivity + Tough-Minded Reserve
1        1   Interpretive Openness + Event-Ready Extraversion
2        2       Flexible Acceptance + Quiet Self-Containment
3        3   Finish-First Focus + Unstructured Under Pressure
4        4        Warmly Intuitive Style + Reserved Composure
5        5  Empathic Sensitivity + Low-initiative Sociability
6        6              Broad Curiosity + Deliberate Planning
\nFirst two clusters (3-sentence descriptions):
Cluster 0 — Focused Selectivity + Tough-Minded Reserve
This cluster combines focused selectivity, tough-minded reserve, and concrete stability as its most distinctive tendencies.
They benefit from flexibility and the ability to adjust to varied contexts.
Potential weaknesses include narrower focus and less appetite for ambiguity, reduced emotional expressiveness.
Cluster 1 — Interpretive Openness + Event-Ready Extraversion
This cluster c

Cluster names & three‑sentence descriptions

The names are formed from the strongest (absolute) factor directions in each cluster’s centroid and are also recorded in the Name column of cluster_centroids_MAP13.csv. 1



Focused Selectivity + Tough‑Minded Reserve
This cluster combines focused selectivity, tough‑minded reserve, and concrete stability as its most distinctive tendencies. They benefit from flexibility and the ability to adjust to varied contexts. Potential weaknesses include narrower focus and less appetite for ambiguity, and reduced emotional expressiveness. 1


Interpretive Openness + Event‑Ready Extraversion
This cluster combines interpretive openness, event‑ready extraversion, and reserved composure as its most distinctive tendencies. Their strengths include open‑minded curiosity and perspective‑taking. Potential weaknesses include lower social initiative in group settings (e.g., not always initiating or sustaining group momentum). 1


Flexible Acceptance + Quiet Self‑Containment
This cluster combines flexible acceptance, quiet self‑containment, and orderliness under pressure as its most distinctive tendencies. Their strengths include reliable execution and follow‑through. Potential blind spots include over‑reliance on preferred styles when contexts demand the opposite. 1


Finish‑First Focus + Unstructured Under Pressure
This cluster combines finish‑first focus, unstructured under pressure, and an analytic (head‑first) style as its most distinctive tendencies. Their strengths include clear analytical thinking alongside reliable execution and follow‑through. Potential weaknesses include inconsistency and uneven follow‑through when demands spike suddenly. 1


Warmly Intuitive Style + Reserved Composure
This cluster combines a warmly intuitive style, reserved composure, and a single‑perspective focus as its most distinctive tendencies. They benefit from flexibility and the ability to adjust to varied contexts. Potential weaknesses include less reliance on structured analysis, lower social initiative in group settings, and a narrower focus with lower appetite for ambiguity. 1


Empathic Sensitivity + Low‑Initiative Sociability
This cluster combines empathic sensitivity, low‑initiative sociability, and elements of expressive sociability as its most distinctive tendencies. Their strengths include interpersonal empathy and sensitivity. Potential weaknesses include lower social initiative in group settings, which can limit group progress unless supported. 1


Broad Curiosity + Deliberate Planning
This cluster combines broad curiosity, deliberate planning, and pro‑social initiative as its most distinctive tendencies. Their strengths include confident, prosocial engagement, open‑minded curiosity and perspective‑taking, and reliable execution and follow‑through. Potential blind spots include over‑reliance on preferred styles when contexts demand the opposite. 1

In [6]:
# === Person 12 profile: factors → cluster → concise summary ===
# Prereqs: pandas, numpy, scikit-learn

import pandas as pd, numpy as np, re
from pathlib import Path
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import FactorAnalysis
from sklearn.cluster import MiniBatchKMeans
from sklearn.metrics import silhouette_score

# ---------- 0) Load & prepare ----------
fn = fn
for enc in ["utf-8", "utf-8-sig", "cp1252", "latin1"]:
    try:
        df = pd.read_csv(fn, encoding=enc)
        break
    except Exception:
        pass

pid_col = "Response Id" if "Response Id" in df.columns else None
label_col = "Personality" if "Personality" in df.columns else None

# Keep only numeric item columns (drop label and id for factor extraction)
X = df.drop(columns=[label_col] if label_col in df.columns else []).copy()
X_items = X.drop(columns=[pid_col]) if pid_col in X.columns else X.copy()
for c in X_items.columns:
    X_items[c] = pd.to_numeric(X_items[c], errors="coerce")
X_items = X_items.apply(lambda s: s.fillna(s.mean()), axis=0)

# Standardize item responses
Z = StandardScaler(with_mean=True, with_std=True).fit_transform(X_items.values)

# ---------- 1) Choose #factors with Velicer’s MAP ----------
R = np.corrcoef(Z, rowvar=False)
vals, vecs = np.linalg.eigh(R)
idx = np.argsort(vals)[::-1]
vals, vecs = vals[idx], vecs[:, idx]
A = vecs @ np.diag(np.sqrt(np.maximum(vals, 0)))

p = R.shape[0]
max_factors = min(30, p-1)
map_vals = []
for m in range(max_factors + 1):
    if m == 0:
        resid = R - np.eye(p)
    else:
        Rm = A[:, :m] @ A[:, :m].T
        resid = R - Rm
        np.fill_diagonal(resid, 0)
    off = resid[~np.eye(p, dtype=bool)]
    map_vals.append(np.mean(off**2))
m_final = int(np.argmin(map_vals)) or 1

# ---------- 2) Factor Analysis + Varimax rotation ----------
def varimax(Phi, gamma=1.0, q=50, tol=1e-6):
    """Orthogonal varimax rotation."""
    p, k = Phi.shape
    Rm = np.eye(k)
    d = 0
    for _ in range(q):
        d_old = d
        Lam = Phi @ Rm
        u, s, vh = np.linalg.svd(Phi.T @ (Lam**3 - (gamma/p) * (Lam @ np.diag(np.diag(Lam.T @ Lam)))))
        Rm = u @ vh
        d = np.sum(s)
        if d_old != 0 and d/d_old < 1 + tol:
            break
    return Phi @ Rm, Rm

fa = FactorAnalysis(n_components=m_final, random_state=123)
fa.fit(Z)
L_unrot = fa.components_.T
L_rot, _ = varimax(L_unrot)

# ---------- 3) Regression factor scores (using rotated loadings) ----------
Psi = np.diag(fa.noise_variance_)
Psi_inv = np.linalg.inv(Psi)
B = Psi_inv @ L_rot @ np.linalg.inv(L_rot.T @ Psi_inv @ L_rot)  # (p x m)
F_scores = Z @ B                                                # (n x m)
Fz = StandardScaler(with_mean=True, with_std=True).fit_transform(F_scores)

# ---------- 4) Cluster selection (silhouette on subsample) + final fit ----------
rng = np.random.RandomState(42)
sub_idx = rng.choice(Fz.shape[0], size=min(10000, Fz.shape[0]), replace=False)
X_sub = Fz[sub_idx]

best_k, best_sil = None, -1
for k in range(2, 9):
    mbk = MiniBatchKMeans(n_clusters=k, random_state=123, batch_size=4096, n_init='auto')
    labels_sub = mbk.fit_predict(X_sub)
    sil = silhouette_score(X_sub, labels_sub)
    if sil > best_sil:
        best_k, best_sil = k, sil

km = MiniBatchKMeans(n_clusters=best_k, random_state=123, batch_size=4096, n_init='auto')
labels = km.fit_predict(Fz)

# ---------- 5) Person 12: factor profile ----------
# Locate the row for Response Id == 12 (fallback to row index 12 if id missing)
if pid_col is not None and 12 in df[pid_col].values:
    irow = df.index[df[pid_col] == 12][0]
else:
    irow = 12  # fallback

p_scores = Fz[irow]    # standardized factor scores (z)
p_cluster = labels[irow]

# Human-readable factor names (consistent with prior analysis)
factor_names = [
    "Quiet Self-Containment",
    "Planner–Improviser Dial",
    "Head-First Analytic Style",
    "Interpretive Openness & Mood Flux",
    "Social Boldness & Expressivity",
    "Order Under Pressure",
    "Wide-Angle Curiosity",
    "Pro-Social Initiative",
    "Finish-First Task Focus",
    "Event-Shy Introversion",
    "Perspective-Taking Spontaneity",
    "Soft-Hearted Sensitivity",
    "Perfectionist Strain",
]

order = np.argsort(p_scores)
weak_idx = order[:3]
strong_idx = order[-3:][::-1]

summary = {
    "Response Id": int(df.loc[irow, pid_col]) if pid_col else irow,
    "Cluster": int(p_cluster),
    "TopStrengths": [(factor_names[i], float(p_scores[i])) for i in strong_idx],
    "TopWeaknesses": [(factor_names[i], float(p_scores[i])) for i in weak_idx],
    "AllScores": {factor_names[i]: float(p_scores[i]) for i in range(len(factor_names))}
}

print(summary)

# ---------- 6) Produce the 3 sentences ----------
line1 = (f"Strengths: very high on {strong_idx.size and factor_names[strong_idx[0]]} "
         f"({p_scores[strong_idx[0]]:+.2f} z), "
         f"{factor_names[strong_idx[1]]} ({p_scores[strong_idx[1]]:+.2f} z), "
         f"and {factor_names[strong_idx[2]]} ({p_scores[strong_idx[2]]:+.2f} z), "
         f"with above-average {factor_names[2]} ({p_scores[2]:+.2f} z).")

line2 = (f"Average/weak spots: near-average on {factor_names[0]} ({p_scores[0]:+.2f} z) "
         f"and {factor_names[9]} ({p_scores[9]:+.2f} z), "
         f"with clear lows on {factor_names[10]} ({p_scores[10]:+.2f} z), "
         f"{factor_names[1]} ({p_scores[1]:+.2f} z), "
         f"and {factor_names[4]} ({p_scores[4]:+.2f} z).")

line3 = ("Conclusion: Person 12 appears caring and reliably prosocial under pressure, "
         "but may prefer structure over improvisation and lower-key social exposure—"
         "benefiting from environments that value empathy and steady execution.")

print(line1)
print(line2)
print(line3)

# Optional: save the three-sentence summary to a text file
with open("person12_summary.txt", "w", encoding="utf-8") as f:
    for ln in (line1, line2, line3):
        f.write(ln + "\n")

{'Response Id': 12, 'Cluster': 1, 'TopStrengths': [('Soft-Hearted Sensitivity', 1.7364135041480548), ('Pro-Social Initiative', 1.0929935137385614), ('Order Under Pressure', 0.9144904470516845)], 'TopWeaknesses': [('Perspective-Taking Spontaneity', -2.105753055473548), ('Planner–Improviser Dial', -1.439194634028493), ('Social Boldness & Expressivity', -0.999534435840154)], 'AllScores': {'Quiet Self-Containment': 0.08536297087924782, 'Planner–Improviser Dial': -1.439194634028493, 'Head-First Analytic Style': 0.7643893053913436, 'Interpretive Openness & Mood Flux': -0.5121473393550987, 'Social Boldness & Expressivity': -0.999534435840154, 'Order Under Pressure': 0.9144904470516845, 'Wide-Angle Curiosity': -0.2933887234767568, 'Pro-Social Initiative': 1.0929935137385614, 'Finish-First Task Focus': -0.6500311968011543, 'Event-Shy Introversion': 0.1297946358568369, 'Perspective-Taking Spontaneity': -2.105753055473548, 'Soft-Hearted Sensitivity': 1.7364135041480548, 'Perfectionist Strain': -0